<img width="8%" alt="Archivist" src="https://upload.wikimedia.org/wikipedia/commons/thumb/e/e1/Biblioth%C3%A8que_Inguimbertine_salle_X.jpg/1920px-Biblioth%C3%A8que_Inguimbertine_salle_X.jpg" style="border-radius: 15%">

# Archivist - Query
<a href="https://github.com/mbasri/archivist">Give Feedback</a> | <a href="https://github.com/mbasri/archivist">Bug report</a>

**Tags:** #archivist #query #search #qdrant #ollama #rag #python

**Author:** [Mohamed BASRI](https://github.com/mbasri)

**Last update:** 2026-04-05 (Created: 2026-04-05)

**Description:** Semantic search against the indexed Qdrant collection. Embeds a text query with Ollama and returns the most relevant chunks with their source file.

**References:**
- [Qdrant Python client](https://python-client.qdrant.tech/)
- [Ollama Python library](https://github.com/ollama/ollama-python)
- [Archivist project](https://github.com/mbasri/archivist)

## Setup

### Install dependencies

In [ ]:
import sys
import subprocess
from pathlib import Path

project_root = Path().resolve()
project_root = project_root.parent if project_root.name == "notebooks" else project_root

result = subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "-r", str(project_root / "requirements.txt")],
    capture_output=True, text=True
)
print(result.stderr[-2000:] if result.returncode != 0 else "✓ Dependencies installed")

## Input

### Import libraries

In [ ]:
import os
import json
from dotenv import load_dotenv
import ollama
from qdrant_client import QdrantClient

load_dotenv(project_root / ".env")
print("✓ .env loaded")

### Setup Variables
- `query`: the question or text to search for
- `top_k`: number of results to return

In [ ]:
query  = "Bonjour"
top_k  = 5

collection_name = os.getenv("COLLECTION_NAME", "archivist")
qdrant_host     = os.getenv("QDRANT_HOST",     "localhost")
qdrant_port     = int(os.getenv("QDRANT_PORT", 6333))
ollama_base_url = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
embedding_model = os.getenv("EMBEDDING_MODEL", "nomic-embed-text")

## Model

### Embed query
Convert the query text into a vector using the same embedding model used during indexing.

In [ ]:
ollama_client = ollama.Client(host=ollama_base_url)
query_vector  = ollama_client.embed(model=embedding_model, input=query)["embeddings"][0]

print(f"Query   : {query}")
print(f"Vector  : {len(query_vector)} dimensions")

### Search in Qdrant

In [ ]:
qdrant  = QdrantClient(host=qdrant_host, port=qdrant_port)
results = qdrant.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=top_k,
    with_payload=True,
).points

## Output

### Display results

In [ ]:
print(f"Query: {query}")
print(f"Top {top_k} results:\n")

for i, result in enumerate(results):
    if "_node_content" not in result.payload:
        continue

    node      = json.loads(result.payload["_node_content"])
    text      = node.get("text", "")
    file_name = result.payload.get("file_name", "unknown")

    print(f"{'─'*60}")
    print(f"#{i+1}  Score : {result.score:.4f}")
    print(f"     File  : {file_name}")
    print(f"     Text  : {text[:300]}...")

print(f"{'─'*60}")